In [3]:
from langgraph.store.postgres import PostgresStore
from typing import Final, Tuple
from rich import print

from dotenv import load_dotenv

load_dotenv(override=True)

import os

DB_URL = os.getenv("DB_URL")

with PostgresStore.from_conn_string(DB_URL) as store:
    #1. 長期記憶ストレージのテーブルを作成
    store.setup()
    #2. 永続記憶データを構築
    #2.1 名前空間を構築
    USERS_NS: Final[Tuple[str]] = ("users",)
    PREFERENCES_KEY: Final[str] = "preferences"

    namespace1 = (*USERS_NS, "Alice")
    namespace2 = (*USERS_NS, "Bob")
    namespace3 = (*USERS_NS, "Black")

    value1 = {
        "course": "コンピュータアーキテクチャ",
        "sports": "ランニング",
        "food": "濃厚ミルクヨーグルト"
    }

    value2 = {
        "course": "デジタル回路とアナログ回路",
        "sports": "ランニング",
        "food": "みたらし団子"
    }

    value3 = {
        "course": "デジタル回路とアナログ回路",
        "sports": "バドミントン",
        "food": "濃厚ミルクヨーグルト"
    }

    #3. 永続記憶データを書き込む
    store.put(namespace1, PREFERENCES_KEY, value1)
    store.put(namespace2, PREFERENCES_KEY, value2)
    store.put(namespace3, PREFERENCES_KEY, value3)

    for item in store.search(USERS_NS):
        print(item)


Item(namespace=['users', 'Black'], key='preferences', value={'food': '濃厚ミルクヨーグルト', 'course': 
'デジタル回路とアナログ回路', 'sports': 'バドミントン'}, created_at='2026-08-15T03:28:07.965916+00:00', 
updated_at='2026-08-15T03:28:07.965916+00:00', score=None)

Item(namespace=['users', 'Bob'], key='preferences', value={'food': 'みたらし団子', 'course': 
'デジタル回路とアナログ回路', 'sports': 'ランニング'}, created_at='2026-08-15T03:28:07.919694+00:00', 
updated_at='2026-08-15T03:28:07.919694+00:00', score=None)

Item(namespace=['users', 'Alice'], key='preferences', value={'food': '濃厚ミルクヨーグルト', 'course': 
'コンピュータアーキテクチャ', 'sports': 'ランニング'}, created_at='2026-08-15T03:28:07.874921+00:00', 
updated_at='2026-08-15T03:28:07.874921+00:00', score=None)

In [ ]:
from typing import TypedDict, Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph.message import MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. グローバル状態を宣言
class OverAllState(MessagesState):
    username: str
    user_input: str
    output: str
    preferences: dict[str, str]


#2. ノードを宣言
#2.1 ルーティング関数 状態にユーザーの好みがなければ長期記憶から検索し、あればそのままLLMノードを実行する
def router(state: OverAllState) -> Literal["check_preference_node", "llm_node"]:
    if not state.get("preferences"):
        logger.info("長期記憶からユーザーの好みを読み込む必要があります")
        return "check_preference_node"
    logger.info("ユーザーの好みは既に存在するため、検索は不要です")
    return "llm_node"


#2.2 長期記憶を確認するノード
def check_preference_node(state: OverAllState, runtime: Runtime) -> OverAllState:
    #1. 名前空間を組み立てる
    username = state["username"]
    namespace = (*USERS_NS, username)
    key = PREFERENCES_KEY
    #2. 長期記憶データを取得
    run_store = runtime.store
    run_item = run_store.get(namespace, key)

    if not item:
        logger.warning("長期記憶に{}の好みデータがありません", username)
        return {}

    logger.info("長期記憶に保存されている{}の好みデータは{}です", username, run_item.value)
    return {
        "preferences": item.value
    }


def llm_node(state: OverAllState) -> OverAllState:
    # 長期記憶が存在するか確認
    preference = state.get("preferences", {})
    user_input = state["user_input"]
    human_prompt = (f"これはユーザーの好みです: {preference}\n、これはユーザーの要望です:{user_input}")
    system_prompt = "ユーザーの好みに基づいて、ユーザーの要望に応えてください"

    messages: list[SystemMessage | HumanMessage | AIMessage | ToolMessage] = [
        SystemMessage(content=system_prompt)] if not state.get("messages", []) else state["messages"]

    model_response = model.invoke(messages + [HumanMessage(content=human_prompt)])

    output = model_response.content

    return {
        "messages": messages + [HumanMessage(content=human_prompt), model_response],
        "output": output
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("check_preference_node", check_preference_node)
builder.add_node("llm_node", llm_node)

builder.add_conditional_edges(START, router, path_map=["check_preference_node", "llm_node"])
builder.add_edge("check_preference_node", "llm_node")
builder.add_edge("llm_node", END)

#4. 長期記憶と短期記憶を構築
with PostgresStore.from_conn_string(DB_URL) as store, PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 冪等な操作 複数回実行してもテーブルは再作成されず、データベース内のデータも削除されない
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer, store=store)

    from IPython.display import display

    display(graph)

    config = {
        "configurable": {"thread_id": "777"}
    }

    res = graph.invoke({"username": "Alice", "user_input": "少し退屈なので、話し相手になってください"}, config=config)

    print('=' * 50)
    print(res)

    res1 = graph.invoke({"username": "Alice", "user_input": "ヨーグルトをおすすめしてください"}, config=config)

    print('=' * 50)
    print(res1)

